In [ ]:
# Install Libraries

!pip install transformers datasets==2.14.6 evaluate seqeval accelerate conllu

In [ ]:
# Import Libraries

import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from evaluate import load

In [ ]:
# Load Dataset

dataset = load_dataset("universal_dependencies", "en_ewt")

In [ ]:
print(dataset)
print(dataset["train"][0])

In [ ]:
# Extract Labels

label_list = dataset["train"].features["upos"].feature.names
num_labels = len(label_list)

print(label_list)

In [ ]:
# Load Tokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# Tokenization

def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        padding=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["upos"][word_idx])
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels)

In [ ]:
# Model Setup

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

In [ ]:
# Training Arguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1
)


In [ ]:
# Evaluation Metric

metric = load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

In [ ]:
# Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
# Train Model

trainer.train()

In [ ]:
# Evaluate Model

trainer.evaluate()

In [ ]:
# Inference

import torch

sentence = "Ayush works at IBM in India"

inputs = tokenizer(
    sentence.split(),
    return_tensors="pt",
    is_split_into_words=True
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model(**inputs).logits
predictions = torch.argmax(outputs, dim=2)

tokens = sentence.split()

for token, pred in zip(tokens, predictions[0]):
    print(token, "->", label_list[pred.item()])

# Comparison :

**Comparision Between POS tagging and chunking:**

1.*POS Tagging:*

• POS tagging involves the process of labeling the parts of speech for
each word within the sentence.

• POS tagging identifies the grammatical label associated with each word like whether it is a noun, verb, adjective, or something else.

• POS tagging operates at the word level.
It is comparatively easier to do compared to chunking.

Example:
Sentence: John works at Google

John → Proper Noun
Works → Verb
Google → Proper Noun


2.*Chunking:*

• Chunking refers to the process of forming phrases out of the words within the sentence.

• For example, chunking may involve forming NP (noun phrase) and VP (verb phrase) from the sentence.

• It operates at the level of phrases, and not single words.
Chunking is comparatively more difficult than POS tagging.

Example:
Sentence: John works at Google

[John] → Noun Phrase (NP)
[Works at Google] → Verb Phrase (VP)

# Report / Blog :

**Report: POS Tagging vs Chunking Using BERT:**

🔹 Introduction

In this report, the task of token classification using a transformer model (BERT) for Part-of-Speech (POS) tagging is discussed. This task helps to gain an insight into applying transformer models for sequence labeling.

🔹 Differences between POS Tagging and Chunking

• POS tagging involves labeling words with POS tags.
Chunking involves labeling chunks or phrases that consist of words.

• POS tagging is a word-level task while chunking is phrase-level.

🔹 Problems Faced

• Label Alignment: Aligning labels with subword tokenization is problematic.

• Subword Tokenization: BERT tokenizes words into subwords.

• Libraries Issues: Faced version compatibility issues.
Padding Problems: Proper padding and data collator are required to handle tensor problems.

🔹 Observations
•BERT is excellent for token classification tasks.
•Properly performing preprocessing step is extremely important.
•Sometimes even the tokenization can significantly influence model performance.
•HuggingFace's trainer package greatly simplifies training and evaluation.

🔹 Conclusion

This project showed how transformer models like BERT can be successfully applied to NLP tasks like POS tagging and chunking.